# Narrative Interference Sweep — Inline Generation

Generates narrative trials on-the-fly and evaluates them. No pre-generated JSON needed.

**How it works:**
- Uses `DotaTrialGenerator` to generate trials at runtime (~700/sec)
- Iterates grid: keys × updates × conditions × trials_per_cell
- Same OOM protection, periodic saves, and resume as the pre-generated version

**To add a new domain:** swap `DotaTrialGenerator` for another generator.

In [ ]:
!pip install -q --upgrade transformers accelerate scipy

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CONFIG — All settings in one place
# ═══════════════════════════════════════════════════════════════════════════

CONFIG = {
    # ── Model ──
    "model_name": "Qwen/Qwen2.5-1.5B-Instruct",

    # ── Generator ──
    "domain": "dota2",

    # ── Grid ──
    "key_levels": [2, 3, 5, 7, 10, 15, 20, 30],
    "update_levels": [1, 3, 5, 10, 20, 25, 30, 35, 40, 45, 50],
    "trials_per_cell": 1,       # per (keys, updates, condition)
    "seed_start": 300000,         # change for independent replication

    # ── Generator overrides (passed to generate_trial) ──
    # Set to None for fully random, or fix for controlled experiments
    "gen_overrides": {
        "tracked_attribute": "gold",
        "attribute_mode": "same",
    },

    # ── Output ──
    "results_dir": "results/narrative_sweep",

    # ── Inference ──
    "max_new_tokens": 30,
    "dtype": "float32",
    "save_every_n": 500,

    # ── HuggingFace auth ──
    "hf_token": None,

    # ── GPU ──
    "gpu_ids": [0],            # e.g. [0] for single GPU, [0,1] for 7B models

    # ── Resume ──
    "resume_from": None,        # path to partial results JSON
}

total_trials = len(CONFIG['key_levels']) * len(CONFIG['update_levels']) * 2 * CONFIG['trials_per_cell']
print(f"Model: {CONFIG['model_name']}")
print(f"Domain: {CONFIG['domain']}")
print(f"Grid: {len(CONFIG['key_levels'])} keys x {len(CONFIG['update_levels'])} updates x 2 conditions x {CONFIG['trials_per_cell']} trials")
print(f"Total trials: {total_trials:,}")
print(f"Generator overrides: {CONFIG['gen_overrides']}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SETUP — Generator + Model
# ═══════════════════════════════════════════════════════════════════════════
import json, os, time, gc, sys
from datetime import datetime, timezone
from pathlib import Path
from collections import defaultdict
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# ── Narrative generator ──
# Option 1: If repo is cloned, add to path
REPO_ROOT = Path(".").resolve()
# Walk up to find narrative_generator/
for _ in range(5):
    if (REPO_ROOT / "narrative_generator").exists():
        break
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from narrative_generator.dota2 import DotaTrialGenerator

# Initialize generator
gen = DotaTrialGenerator()
print(f"Generator ready: {gen.__class__.__name__}")

# Quick sanity check
t = gen.generate_trial(2, 3, "RI", seed=42)
gen.validate_trial(t)
print(f"Sanity check passed ({len(t['narrative'])} chars)")

# ── Model ──
MODEL_NAME = CONFIG["model_name"]
MODEL_SHORT = MODEL_NAME.split("/")[-1]
TS = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

if CONFIG["hf_token"]:
    os.environ["HF_TOKEN"] = CONFIG["hf_token"]

RESULTS_DIR = Path(CONFIG["results_dir"]) / MODEL_SHORT
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── GPU selection ──
gpu_ids = CONFIG.get("gpu_ids", [0])
os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(str(g) for g in gpu_ids)
print(f"Using GPUs: {gpu_ids}")

dtype_map = {"float16": torch.float16, "bfloat16": torch.bfloat16, "float32": torch.float32}
print(f"\nLoading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=dtype_map[CONFIG["dtype"]],
    device_map="auto", trust_remote_code=True,
)
model.eval()
ctx_limit = getattr(model.config, "max_position_embeddings", 8192)

if torch.cuda.is_available():
    total_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    allocated = torch.cuda.memory_allocated(0) / 1e9
    free_for_kv = total_mem - allocated - 0.5
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {total_mem:.1f} GB total, {allocated:.1f} GB model, ~{free_for_kv:.1f} GB free")
else:
    free_for_kv = 4.0
    print(f"No CUDA — using {model.device}")
print(f"Context limit: {ctx_limit}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# INFERENCE ENGINE
# ═══════════════════════════════════════════════════════════════════════════

def build_prompt(narrative, question):
    return (
        f"Read the following passage carefully.\n\n"
        f"{narrative}\n\n"
        f"Based ONLY on the passage above, answer the following question "
        f"with a short, exact answer (just the value, no explanation).\n\n"
        f"Question: {question}\nAnswer:"
    )


def format_chat(prompt):
    if hasattr(tokenizer, 'apply_chat_template') and tokenizer.chat_template:
        messages = [
            {"role": "system", "content": "Answer with ONLY the exact value. No explanation."},
            {"role": "user", "content": prompt},
        ]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return prompt


def classify_answer(answer, expected, all_values=None):
    a, e = answer.lower().strip(), expected.lower().strip()
    # Exact or substring match
    if e in a or a in e:
        return "correct"
    # Check if answer matches any other tracked value
    if all_values:
        for v in all_values:
            if v.lower() in a or a in v.lower():
                return "wrong_value"
    return "garbage"


def run_one(trial, condition):
    """Generate prompt from trial, run inference, return result dict."""
    q = trial["questions"][condition]
    expected = q["expected_answer"]
    tracking_key = f"{q['target_entity']} / {q['target_attribute']}"
    all_values = trial["entity_tracking"].get(tracking_key, [])

    prompt = build_prompt(trial["narrative"], q["question"])
    formatted = format_chat(prompt)
    input_ids = tokenizer.encode(formatted, return_tensors="pt")
    n_tokens = input_ids.shape[1]

    base = {
        "seed": trial["config"]["seed"], "condition": condition,
        "expected": expected, "input_tokens": n_tokens,
        "num_keys": trial["num_keys"], "num_updates": trial["num_updates"],
    }

    if n_tokens > ctx_limit * 0.95:
        return {**base, "answer": None, "correct": None, "error_type": "skipped_context"}

    input_ids = input_ids.to(model.device)
    try:
        with torch.no_grad():
            gen_ids = model.generate(
                input_ids, max_new_tokens=CONFIG["max_new_tokens"],
                do_sample=False, pad_token_id=tokenizer.pad_token_id,
            )
        new_ids = gen_ids[0, input_ids.shape[1]:]
        answer = tokenizer.decode(new_ids, skip_special_tokens=True).strip().split("\n")[0].strip()
        del input_ids, gen_ids
    except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
        if "out of memory" in str(e).lower() or isinstance(e, torch.cuda.OutOfMemoryError):
            del input_ids
            if torch.cuda.is_available(): torch.cuda.empty_cache()
            gc.collect()
            return {**base, "answer": "[OOM]", "correct": False, "error_type": "oom"}
        raise

    error_type = classify_answer(answer, expected, all_values)
    return {**base, "answer": answer, "correct": error_type == "correct", "error_type": error_type}


def bootstrap_ci(data, n_bootstrap=2000, ci=0.95):
    if not data: return 0.0, 0.0, 0.0
    arr = np.array(data, dtype=float)
    mean = arr.mean()
    if len(arr) < 3: return mean, 0.0, 1.0
    rng = np.random.RandomState(42)
    boot = [rng.choice(arr, size=len(arr), replace=True).mean() for _ in range(n_bootstrap)]
    alpha = (1 - ci) / 2
    return mean, np.percentile(boot, alpha * 100), np.percentile(boot, (1 - alpha) * 100)


def save_results(output, partial=False):
    suffix = '_partial' if partial else ''
    path = RESULTS_DIR / f"narrative_inline_{TS}{suffix}.json"
    with open(path, 'w') as f:
        json.dump(output, f, indent=2)
    print(f"  -> Saved to {path}")
    return path


print("Engine ready.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# RUN SWEEP — Generate + Evaluate inline
# ═══════════════════════════════════════════════════════════════════════════

key_levels = CONFIG["key_levels"]
update_levels = CONFIG["update_levels"]
trials_per_cell = CONFIG["trials_per_cell"]
seed = CONFIG["seed_start"]
overrides = CONFIG.get("gen_overrides", {}) or {}

# Resume support
results_list = []
completed_seeds = set()
if CONFIG["resume_from"] and os.path.exists(CONFIG["resume_from"]):
    with open(CONFIG["resume_from"]) as f:
        prev = json.load(f)
    results_list = prev.get("results", [])
    completed_seeds = {(r["seed"], r["condition"]) for r in results_list}
    print(f"Resuming: {len(completed_seeds)} trials already done")

correct = {"RI": 0, "PI": 0}
total_done = {"RI": 0, "PI": 0}
skipped = 0
oom_count = 0
gen_failures = 0
start_time = time.time()
save_every = CONFIG["save_every_n"]
trials_since_save = 0

total_cells = len(key_levels) * len(update_levels) * 2
total_trials = total_cells * trials_per_cell
trial_idx = 0

for nk in key_levels:
    for nu in update_levels:
        for condition in ["RI", "PI"]:
            for t in range(trials_per_cell):
                trial_idx += 1

                if (seed, condition) in completed_seeds:
                    seed += 1
                    continue

                # ── Generate trial on-the-fly ──
                try:
                    trial = gen.generate_trial(nk, nu, condition, seed, **overrides)
                except Exception as e:
                    gen_failures += 1
                    seed += 1
                    continue

                # ── Run inference ──
                result = run_one(trial, condition)
                results_list.append(result)
                trials_since_save += 1

                if result["error_type"] == "skipped_context":
                    skipped += 1
                elif result["error_type"] == "oom":
                    oom_count += 1
                else:
                    total_done[condition] += 1
                    if result["correct"]:
                        correct[condition] += 1

                # Progress
                if trial_idx % 20 == 0:
                    ri_acc = correct["RI"] / max(total_done["RI"], 1)
                    pi_acc = correct["PI"] / max(total_done["PI"], 1)
                    elapsed = time.time() - start_time
                    done = total_done["RI"] + total_done["PI"] + skipped + oom_count
                    remaining = total_trials - trial_idx
                    rate = done / max(elapsed, 1)
                    eta_min = remaining / max(rate, 0.01) / 60
                    print(f"  [{trial_idx}/{total_trials}] {nk}k_{nu}u "
                          f"RI={ri_acc:.0%} PI={pi_acc:.0%} "
                          f"({rate:.1f}/sec, skip={skipped}, oom={oom_count}) "
                          f"[ETA: {eta_min:.0f}min]")

                # Periodic save
                if trials_since_save >= save_every:
                    partial_output = {
                        "metadata": {
                            "model": MODEL_NAME, "timestamp": TS,
                            "domain": CONFIG["domain"],
                            "grid": {"num_keys": key_levels, "num_updates": update_levels},
                            "trials_per_cell": trials_per_cell,
                            "gen_overrides": overrides,
                            "seed_start": CONFIG["seed_start"],
                        },
                        "results": results_list,
                    }
                    save_results(partial_output, partial=True)
                    trials_since_save = 0

                # Clear cache periodically
                if trial_idx % 50 == 0 and torch.cuda.is_available():
                    torch.cuda.empty_cache()

                seed += 1

elapsed = time.time() - start_time
print(f"\nDone in {elapsed/60:.1f} min ({elapsed:.0f}s)")
print(f"Processed: {total_done['RI'] + total_done['PI']} | Skipped: {skipped} | OOM: {oom_count} | Gen failures: {gen_failures}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# RESULTS SUMMARY
# ═══════════════════════════════════════════════════════════════════════════

valid = [r for r in results_list if r.get("correct") is not None]
ri_r = [r for r in valid if r["condition"] == "RI"]
pi_r = [r for r in valid if r["condition"] == "PI"]

ri_acc = sum(r["correct"] for r in ri_r) / max(len(ri_r), 1)
pi_acc = sum(r["correct"] for r in pi_r) / max(len(pi_r), 1)
ri_mean, ri_lo, ri_hi = bootstrap_ci([r["correct"] for r in ri_r])
pi_mean, pi_lo, pi_hi = bootstrap_ci([r["correct"] for r in pi_r])

print(f"{'='*60}")
print(f"NARRATIVE SWEEP (inline) — {MODEL_SHORT}")
print(f"{'='*60}")
print(f"  RI: {ri_acc:.1%} [{ri_lo:.1%}-{ri_hi:.1%}] ({sum(r['correct'] for r in ri_r)}/{len(ri_r)})")
print(f"  PI: {pi_acc:.1%} [{pi_lo:.1%}-{pi_hi:.1%}] ({sum(r['correct'] for r in pi_r)}/{len(pi_r)})")
print(f"  PI > RI: {pi_acc > ri_acc} (diff = {pi_acc - ri_acc:+.1%})")
print(f"  Skipped: {skipped} | OOM: {oom_count} | Gen failures: {gen_failures}")

# Grid breakdown
cell_stats = defaultdict(lambda: {"RI": {"c": 0, "t": 0}, "PI": {"c": 0, "t": 0}})
for r in valid:
    cell = f"{r['num_keys']}k_{r['num_updates']}u"
    cell_stats[cell][r["condition"]]["t"] += 1
    if r["correct"]: cell_stats[cell][r["condition"]]["c"] += 1

print(f"\nGrid (RI / PI):")
# Header
print(f"  {'':>6}", end="")
for u in update_levels:
    print(f"  {u:>3}u      ", end="")
print()

for k in key_levels:
    row_ri = f"  {k:>4}k "
    row_pi = f"  {'':>5} "
    for u in update_levels:
        cell = f"{k}k_{u}u"
        if cell in cell_stats:
            cs = cell_stats[cell]
            ri_a = cs['RI']['c'] / max(cs['RI']['t'], 1) * 100
            pi_a = cs['PI']['c'] / max(cs['PI']['t'], 1) * 100
            diff = pi_a - ri_a
            row_ri += f" {ri_a:4.0f}/{pi_a:<4.0f}"
            row_pi += f" {diff:+5.0f}pp  "
        else:
            row_ri += f"   --/--  "
            row_pi += f"     --   "
    print(row_ri)

# Errors
err = defaultdict(lambda: defaultdict(int))
for r in valid: err[r["condition"]][r["error_type"]] += 1
print(f"\nErrors:")
for c in ["RI", "PI"]: print(f"  {c}: {dict(err[c])}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SAVE FINAL
# ═══════════════════════════════════════════════════════════════════════════

final = {
    "metadata": {
        "model": MODEL_NAME, "model_short": MODEL_SHORT,
        "domain": CONFIG["domain"],
        "timestamp": TS,
        "grid": {"num_keys": key_levels, "num_updates": update_levels},
        "trials_per_cell_per_condition": trials_per_cell,
        "gen_overrides": overrides,
        "seed_range": f"{CONFIG['seed_start']}-{seed - 1}",
        "context_limit": ctx_limit,
        "config": CONFIG,
    },
    "summary": {
        "ri_accuracy": ri_acc, "pi_accuracy": pi_acc,
        "ri_ci": [ri_lo, ri_hi], "pi_ci": [pi_lo, pi_hi],
        "ri_n": len(ri_r), "pi_n": len(pi_r),
        "skipped": skipped, "oom": oom_count, "gen_failures": gen_failures,
        "pi_gt_ri": pi_acc > ri_acc, "diff": pi_acc - ri_acc,
    },
    "cell_stats": {k: v for k, v in cell_stats.items()},
    "error_distribution": {k: dict(v) for k, v in err.items()},
    "results": results_list,
}

path = save_results(final, partial=False)
print(f"\nFinal: {path} ({path.stat().st_size / 1024 / 1024:.1f} MB)")